In [2]:
import numpy as np
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm



#a)Numerical Engine & Theoretical Baseline (Validation)

mu = 0.05
sigma = 0.2
S0 = 100
T=1 # 1Year
N=252 #Number of trading days
dt = T/N

S=np.zeros((10000,253)) #Create an empty matrix to fit my data size

S[:,0]=S0

#1)Define the Euler Maruyama function
def EulerMar():  
    for t in range(N):
        S[:,t+1] = S[:,t] + mu * S[:,t] * dt + sigma * S[:,t] * np.sqrt(dt) * np.random.normal(0,1,10000)
EulerMar()

average_numerical = np.mean(S[:,-1])  # -1 because we want the last column. So we go backwards
        
#2)Using Ito's Lemma on f(x)=lnx and solve it. Calculate 10000 times and take mean

S_analytical=S0 * np.exp ((mu-(1/2)*(sigma)**2) * T + sigma * np.sqrt(T) * np.random.normal(0,1,10000))

average_analytical = np.mean(S_analytical)

#3) Take first the expectation of the function and calculate once.

theoretical_solution = (S0) * np.exp(mu*T)

print("Average numerical =", f"{np.mean(S[:, -1]):.3f}", 
      "Average analytical =", f"{average_analytical:.3f}",
      "Theoretical solution =", f"{theoretical_solution:.3f}")


#B)Empirical Data & Volatility Forecasting

raw_prices = yf.download("NVO",start ="2021-06-20",end = "2026-06-20")


#We take adjuded prices because from a)stock splits and b)divident payments, the stock price might drop overnight.
#1) For that we use adjustment on the past values we get the adjusted closing values.Proportionally
#2) For that we cannot just substract like with stock splits.We create a factor (S_old)/(S_new) to do it proportionally.

log_returns = np.diff(np.log(prices))

mu = np.mean(log_returns) * N # Calculate the mean μ=0.03
sigma = np.std(log_returns) * np.sqrt(N) # σ=0.39


r=0.04 # Derived from Treasury's 1 year yield bond
S0 = prices.iloc[-1] 
K_values=[(S0-0.15*S0),S0,(S0+0.15*S0)] # Choose K to be ATM and minus/plus 15%

def Black_Scholes(S0,K,r,sigma,T):
    d1=((np.log(S0/K)+(r+(sigma**2)/2)*T))/(sigma*np.sqrt(T))
    d2=d1-sigma*np.sqrt(T)
    return S0*norm.cdf(d1)-(np.exp(-r*T))*K*norm.cdf(d2)

BS_Call=[Black_Scholes(S0,K,r,sigma,T) for K in K_values]

def Monte_Carlo(S0,r,sigma,T,K):
    z=np.random.normal(0,1,10000)
    ST=S0*np.exp(((r-(sigma**2)/2))*T+sigma*np.sqrt(T)*z)
    Payoff_Call=np.maximum(ST-K,0)
    Expected_payoff_Call=np.exp(-r*T)*Payoff_Call
    Call_Option_Price=np.mean(Expected_payoff_Call)
    return Call_Option_Price

MC_Call_Price=[Monte_Carlo(S0,r,sigma,T,K) for K in K_values]
df=pd.DataFrame(index=['BS Call','MC Call'],columns=K_values,data=np.vstack([BS_Call,MC_Call_Price])).round(3)
print(df)

[*********************100%***********************]  1 of 1 completed

Average numerical = 104.762 Average analytical = 105.240 Theoretical solution = 105.127
         36.711499  43.189999  49.668498
BS Call     10.807      7.422      4.983
MC Call     10.710      7.369      4.903
